# Preprocessing

In [29]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

## load data

In [ ]:
# load the LLM labeled dataset
df_llm = pd.read_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/annotations.parquet")

# load the full dataframe without label 
df = pd.read_csv("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/raw/climate_misinfo_2023-2025.csv")

## cleaning

In [9]:
# rename the label column
df_llm.rename(columns={"categories": "CardsSubClaim"}, inplace=True)

In [10]:
# add an extra label column to the LLM labeled dataset
df_llm["CardsClaim"] = df_llm["CardsSubClaim"].apply(
    lambda arr: sorted({f"{str(x).split('_')[0]}" for x in arr})
    if isinstance(arr, (list, np.ndarray)) else []
)

df_llm.head(2)

,ArticleKey,CardsSubClaim,prompt_tokens,completion_tokens,CardsClaim
0,ea4d58f9,[1_7_0],4401,41,[1]
1,ead59516,[1_7_0],4570,41,[1]


In [11]:
# remove redundant columns
columns_to_drop = ["completion_tokens", "prompt_tokens"]

df_llm = df_llm.drop(columns=columns_to_drop)

df_llm.head(2)

,ArticleKey,CardsSubClaim,CardsClaim
0,ea4d58f9,[1_7_0],[1]
1,ead59516,[1_7_0],[1]


In [12]:
# merge the annoatations with the original dataframe
df = df.merge(df_llm, on="ArticleKey")

df.head(2)

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,CardsSubClaim,CardsClaim
0,e9e57e17,2023-09-30 07:25:41,Søløve smagte kortvarigt friheden i oversvømme...,NaN,En skolebus forsøger her at komme gennem vandm...,303,da,NaN,180Grader.dk,4R2,...,Webkilder,2023-09-30,9,2023,Søløve smagte kortvarigt friheden i oversvømme...,Negative,-1.0,True,[1_7_0],[1]
1,e9620bc1,2023-02-02 07:51:09,HVEM SKAL NU BETALE?,NaN,Af Asger Aamund. Udgivet på Tiggeri plejede a...,866,NaN,NaN,24nyt.dk,3BI,...,Webkilder,2023-02-02,2,2023,HVEM SKAL NU BETALE? Af Asger Aamund. Udgivet ...,Negative,-1.0,False,"[4_2_4, 3_5_0, 4_1_1_2]","[3, 4]"


In [13]:
# check label variation 
unique_labels = set(label for labels in df["CardsClaim"] for label in labels)
print(sorted(unique_labels))

['0', '1', '12', '2', '3', '4', '5', '6', '7']


In [14]:
# examine occurences of label 12
df[df["CardsClaim"].apply(lambda x: "12" in x)]

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,CardsSubClaim,CardsClaim
10747,ea5abac3,2024-08-18 00:00:00,"Føler du, at verden er af lave? Her er 19 gode...",Jyllands-Posten har ringet til 15 danske ekspe...,1DE GLOBALE CO2-UDLEDNINGER TOPPER MÅSKE SNART...,1319,da,LASSE MOMME - BENJAMIN JUNGDAL THORGERD JEN...,Jyllands-Posten,JYP,...,Landsdækkende dagblade,2024-08-18,8,2024,"Føler du, at verden er af lave? Her er 19 gode...",Neutral,0.0,False,"[12_0_0, 4_2_0]","[12, 4]"


In [15]:
# remove single observation with invalid label 
df = df[~df["CardsClaim"].apply(lambda x: "12" in x)]

In [16]:
# check labels after cleaning
label_counts = Counter(label for labels in df["CardsClaim"] for label in labels)

print(label_counts)

Counter({'4': 14750, '3': 2871, '1': 2827, '0': 911, '7': 789, '6': 536, '2': 473, '5': 337})


In [17]:
# convert labels to integers
df["CardsClaim"] = df["CardsClaim"].apply(
    lambda labels: [int(l) for l in labels]
)

In [18]:
# convert labels to one hot encodings
mlb = MultiLabelBinarizer(classes=range(8))

Y = mlb.fit_transform(df["CardsClaim"])

In [19]:
# name new columns for each label 
label_cols = [f"label_{i}" for i in range(8)]

# apply to dataframe
df[label_cols] = Y

In [20]:
# see result
df.head(2)

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,CardsSubClaim,CardsClaim,label_0,label_1,label_2,label_3,label_4,label_5,label_6,label_7
0,e9e57e17,2023-09-30 07:25:41,Søløve smagte kortvarigt friheden i oversvømme...,NaN,En skolebus forsøger her at komme gennem vandm...,303,da,NaN,180Grader.dk,4R2,...,[1_7_0],[1],0,1,0,0,0,0,0,0
1,e9620bc1,2023-02-02 07:51:09,HVEM SKAL NU BETALE?,NaN,Af Asger Aamund. Udgivet på Tiggeri plejede a...,866,NaN,NaN,24nyt.dk,3BI,...,"[4_2_4, 3_5_0, 4_1_1_2]","[3, 4]",0,0,0,1,1,0,0,0


In [21]:
# check that label distribution is unchanged
df[label_cols].sum()

label_0      911
label_1     2827
label_2      473
label_3     2871
label_4    14750
label_5      337
label_6      536
label_7      789
dtype: int64

## save data for training

In [ ]:
# save clean dataframe before splitting
df.to_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/climate_misinfo_clean.parquet")

In [30]:
# split data into training and test
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in msss.split(df, df[label_cols]):
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]


# extract validation set from training set
msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, val_idx in msss_val.split(train_df, train_df[label_cols]):
    train_final = train_df.iloc[train_idx]
    val_df = train_df.iloc[val_idx]

In [ ]:
# check label distribution 
def label_distribution(df, label_cols):
    return df[label_cols].mean()

print("Full:\n", label_distribution(df, label_cols))
print("\nTrain:\n", label_distribution(train_final, label_cols))
print("\nVal:\n", label_distribution(val_df, label_cols))
print("\nTest:\n", label_distribution(test_df, label_cols))

Full:
 label_0    0.050682
label_1    0.157274
label_2    0.026314
label_3    0.159722
label_4    0.820584
label_5    0.018748
label_6    0.029819
label_7    0.043894
dtype: float64

Train:
 label_0    0.050722
label_1    0.157473
label_2    0.026275
label_3    0.159910
label_4    0.821298
label_5    0.018792
label_6    0.029842
label_7    0.043936
dtype: float64

Val:
 label_0    0.050694
label_1    0.156944
label_2    0.026389
label_3    0.159375
label_4    0.819444
label_5    0.018750
label_6    0.029861
label_7    0.043750
dtype: float64

Test:
 label_0    0.050542
label_1    0.156901
label_2    0.026382
label_3    0.159400
label_4    0.819217
label_5    0.018606
label_6    0.029714
label_7    0.043877
dtype: float64


In [ ]:
# save splits 
train_final.to_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/train.parquet")
val_df.to_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/val.parquet")
test_df.to_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/test.parquet")